# 02 · Modelado y experimentación (Google Colab + GPU)

**Fase 2 (CRISP-DM: Preparación + Modelado + Evaluación)**

Entrena y compara **tres** modelos sobre el dataset limpio (`data/processed`):

1. **CNN desde cero** — baseline de referencia.
2. **MobileNetV2** (transfer learning) — ligero, candidato a desplegar.
3. **EfficientNetB0** (transfer learning) — comparación.

> ⚙️ **Ejecuta en Colab con GPU:** `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.

> 🧭 **Métrica norte:** Recall de la clase *fractura* + AUC. Accuracy es secundaria.

## 1 · Código del proyecto en Colab

**Opción A (recomendada) — repo ya está en GitHub público:**

In [1]:
import os, pathlib, sys

# Detectar entorno: Colab tiene /content y el módulo google.colab
_in_colab = 'google.colab' in sys.modules or pathlib.Path('/content').exists()

if _in_colab:
    REPO_URL  = "https://github.com/stevenrq/fracturas-rayos-x.git"
    REPO_PATH = "/content/fracturas-rayos-x"
    if not pathlib.Path(REPO_PATH).exists():
        !git clone $REPO_URL $REPO_PATH
    else:
        !git -C $REPO_PATH pull origin main
    if not pathlib.Path(REPO_PATH).exists():
        raise RuntimeError(f"El repo no existe en {REPO_PATH}. Revisa la URL.")
    os.chdir(REPO_PATH)
else:
    # Local: el repo ya existe; sube hasta encontrar la raíz (donde está src/)
    _here = pathlib.Path.cwd()
    _repo = _here if (_here / 'src').exists() else _here.parent
    REPO_PATH = str(_repo)
    os.chdir(_repo)

print("Entorno:", 'Colab' if _in_colab else 'Local')
print("cwd:", os.getcwd())


From https://github.com/stevenrq/fracturas-rayos-x
 * branch            main       -> FETCH_HEAD
Already up to date.
Entorno: Colab
cwd: /content/fracturas-rayos-x


    **Opción B — el repo aún no está en GitHub (o es privado):** comprime solo la
carpeta `src/` en tu máquina y súbela:

```bash
# En la raíz del repo (Ubuntu):
zip -r src.zip src/
```
Luego ejecuta la siguiente celda:

In [2]:
# --- Opción B: subir src.zip manualmente ---
# Descomenta si usas esta opción (comenta la celda de git clone de arriba).
# from google.colab import files
# files.upload()          # selecciona src.zip
# import zipfile
# with zipfile.ZipFile("src.zip") as z:
#     z.extractall(".")
# print("src/ extraído:", list(Path("src").iterdir()))

## 2 · Dataset en Colab

`data/processed/` no está en GitHub (pesado, va en `.gitignore`). Súbelo desde Drive
o directamente.

Para crear el zip en tu máquina (raíz del repo):
```bash
cd data && zip -r processed.zip processed/ && cd ..
```

In [3]:
import subprocess, sys, pathlib

IMG_EXTS = {".jpg", ".jpeg", ".png"}

def _has_images(folder):
    p = pathlib.Path(folder)
    return p.exists() and any(f.is_file() for f in p.rglob("*") if f.suffix.lower() in IMG_EXTS)

# 1) data/raw/ — descarga vía kagglehub si no existe
if not _has_images("data/raw"):
    print("data/raw/ no encontrado — ejecutando src/prepare_data.py …")
    r = subprocess.run([sys.executable, "src/prepare_data.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        raise RuntimeError(
            "prepare_data.py falló. Revisa las credenciales de Kaggle:\n"
            "https://github.com/Kaggle/kagglehub#authentication"
        )
    print("✓ data/raw/ listo.")
else:
    print("✓ data/raw/ ya existe, omitiendo descarga.")

# 2) data/processed/ — deduplica y re-divide si no existe
if not _has_images("data/processed"):
    print("data/processed/ no encontrado — ejecutando src/deduplicate_split.py …")
    r = subprocess.run([sys.executable, "src/deduplicate_split.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        raise RuntimeError("deduplicate_split.py falló.")
    print("✓ data/processed/ listo.")
else:
    print("✓ data/processed/ ya existe, omitiendo.")

print("Imágenes en train:",
      sum(1 for f in pathlib.Path("data/processed/train").rglob("*") if f.is_file()))

✓ data/raw/ ya existe, omitiendo descarga.
✓ data/processed/ ya existe, omitiendo.
Imágenes en train: 2738


## 3 · Imports y configuración

In [4]:
import sys, pathlib, os

# Ruta fija en Colab; fallback al cwd para ejecución local
REPO = pathlib.Path("/content/fracturas-rayos-x")
if not REPO.exists():
    REPO = pathlib.Path.cwd()

os.chdir(REPO)  # garantiza que el cwd sea la raíz del repo

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("REPO:", REPO)
print("SRC en sys.path:", SRC.exists())


REPO: /content/fracturas-rayos-x
SRC en sys.path: True


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import config, data, model, train, evaluate

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))
print("DATA_DIR:", config.DATA_DIR, "->", config.DATA_DIR.exists())

TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
DATA_DIR: /content/fracturas-rayos-x/data/processed -> True


## 4 · Sanity check de datos y pesos de clase

In [6]:
# Elimina imágenes que TF/libjpeg no puede decodificar.
# Debe correr ANTES de make_datasets() para evitar errores en el pipeline.
import tensorflow as tf, pathlib

IMG_EXTS = {'.jpg', '.jpeg', '.png'}
data_dir = pathlib.Path('data/processed')
all_imgs = [p for p in data_dir.rglob('*') if p.suffix.lower() in IMG_EXTS]
print(f'Escaneando {len(all_imgs)} imágenes con el decodificador TF...')
corrupt = []
for p in all_imgs:
    try:
        raw = tf.io.read_file(str(p))
        tf.io.decode_image(raw, channels=3, expand_animations=False)
    except Exception:
        corrupt.append(p)
        p.unlink()
print(f'Eliminadas {len(corrupt)} imágenes corruptas. Pipeline listo.')


Escaneando 3888 imágenes con el decodificador TF...
Eliminadas 5 imágenes corruptas. Pipeline listo.


In [7]:
train_ds, val_ds, test_ds, class_names = data.make_datasets()
print("Orden de clases:", class_names, " (fractura debe ser la clase 1)")

class_weight = data.compute_class_weights()
print("class_weight:", class_weight)

for xb, yb in train_ds.take(1):
    print("batch imágenes:", xb.shape, xb.dtype, "| etiquetas:", yb.shape)
    break

Found 2735 files belonging to 2 classes.
Found 585 files belonging to 2 classes.
Found 563 files belonging to 2 classes.
Orden de clases: ['0_normal', '1_fracture']  (fractura debe ser la clase 1)
class_weight: {0: 0.8504353233830846, 1: 1.2133984028393967}
batch imágenes: (32, 224, 224, 3) <dtype: 'float32'> | etiquetas: (32, 1)


## 5 · Entrenamiento de los tres modelos

Cada llamada entrena en dos etapas y guarda el mejor en `models/best_<modelo>.keras`.
Ajusta los *epochs* según el tiempo disponible en Colab.

In [8]:
# Crea la carpeta models/ si no existe
(REPO / "models").mkdir(exist_ok=True)
(REPO / "reports" / "figures").mkdir(parents=True, exist_ok=True)

In [9]:
# --- Modelo A: CNN desde cero ---
cnn_net, cnn_ckpt, cnn_hist = train.train("cnn", epochs_head=25, epochs_ft=0, dropout=0.3)

Found 2735 files belonging to 2 classes.
Found 585 files belonging to 2 classes.
Found 563 files belonging to 2 classes.
Orden de clases: ['0_normal', '1_fracture'] (fractura = 1)
class_weight: {0: 0.8504353233830846, 1: 1.2133984028393967}

=== Etapa 1: feature extraction (cnn) ===
Epoch 1/25
85/85 ━━━━━━━━━━━━━━━━━━━━ 25s 132ms/step - accuracy: 0.6963 - auc: 0.7468 - loss: 0.5936 - precision: 0.6247 - recall: 0.6589 - val_accuracy: 0.5829 - val_auc: 0.8190 - val_loss: 0.6689 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/25
 1/85 ━━━━━━━━━━━━━━━━━━━━ 8s 104ms/step - accuracy: 0.7812 - auc: 0.7591 - loss: 0.5352 - precision: 0.8000 - recall: 0.6154

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7812 - auc: 0.7591 - loss: 0.5352 - precision: 0.8000 - recall: 0.6154 - val_accuracy: 0.5829 - val_auc: 0.7842 - val_loss: 0.6697 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/25
85/85 ━━━━━━━━━━━━━━━━━━━━ 10s 112ms/step - accuracy: 0.7225 - auc: 0.7818 - loss: 0.5590 - precision: 0.6591 - recall: 0.6768 - val_accuracy: 0.5812 - val_auc: 0.8232 - val_loss: 0.7174 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/25
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6875 - auc: 0.7065 - loss: 0.6129 - precision: 0.6154 - recall: 0.6154 - val_accuracy: 0.5812 - val_auc: 0.8335 - val_loss: 0.7355 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/25
85/85 ━━━━━━━━━━━━━━━━━━━━ 10s 112ms/step - accuracy: 0.7584 - auc: 0.8158 - loss: 0.5185 - precision: 0.7074 - recall: 0.7067 - val_accuracy: 0.5829 - val_auc: 0.8766 - val_loss: 1

In [10]:
# --- Modelo B: MobileNetV2 (candidato a desplegar) ---
mnet_net, mnet_ckpt, mnet_hist = train.train("mobilenet", epochs_head=15, epochs_ft=10,
                                             fine_tune_at=100, dropout=0.3)

Found 2735 files belonging to 2 classes.
Found 585 files belonging to 2 classes.
Found 563 files belonging to 2 classes.
Orden de clases: ['0_normal', '1_fracture'] (fractura = 1)
class_weight: {0: 0.8504353233830846, 1: 1.2133984028393967}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== Etapa 1: feature extraction (mobilenet) ===
Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.6804 - auc: 0.7478 - loss: 0.5989 - precision: 0.6025 - recall: 0.6598 - val_accuracy: 0.6786 - val_auc: 0.7374 - val_loss: 0.5758 - val_precision: 0.6250 - val_recall: 0.5738 - learning_rate: 0.0010
Epoch 2/15
 1/85 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.7812 - auc: 0.8704 - loss: 0.4457 - precision: 0.6875 - recall: 0.8462

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7812 - auc: 0.8704 - loss: 0.4457 - precision: 0.6875 - recall: 0.8462 - val_accuracy: 0.6855 - val_auc: 0.7384 - val_loss: 0.5741 - val_precision: 0.6364 - val_recall: 0.5738 - learning_rate: 0.0010
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 6s 69ms/step - accuracy: 0.8113 - auc: 0.8894 - loss: 0.4209 - precision: 0.7640 - recall: 0.7846 - val_accuracy: 0.7077 - val_auc: 0.7751 - val_loss: 0.5397 - val_precision: 0.6622 - val_recall: 0.6107 - learning_rate: 0.0010
Epoch 4/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7500 - auc: 0.8502 - loss: 0.4643 - precision: 0.6923 - recall: 0.6923 - val_accuracy: 0.7060 - val_auc: 0.7755 - val_loss: 0.5380 - val_precision: 0.6636 - val_recall: 0.5984 - learning_rate: 0.0010
Epoch 5/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.8487 - auc: 0.9243 - loss: 0.3588 - precision: 0.8129 - recall: 0.8224 - val_accuracy: 0.7094 - val_auc: 0.7777 - val_loss: 0.5403 - val_precision: 0.7

In [11]:
# --- Modelo C: EfficientNetB0 ---
eff_net, eff_ckpt, eff_hist = train.train("efficientnet", epochs_head=15, epochs_ft=10,
                                          fine_tune_at=150, dropout=0.3)

Found 2735 files belonging to 2 classes.
Found 585 files belonging to 2 classes.
Found 563 files belonging to 2 classes.
Orden de clases: ['0_normal', '1_fracture'] (fractura = 1)
class_weight: {0: 0.8504353233830846, 1: 1.2133984028393967}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== Etapa 1: feature extraction (efficientnet) ===
Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 28s 145ms/step - accuracy: 0.7381 - auc: 0.7991 - loss: 0.5553 - precision: 0.6897 - recall: 0.6625 - val_accuracy: 0.6821 - val_auc: 0.7817 - val_loss: 0.5418 - val_precision: 0.6169 - val_recall: 0.6270 - learning_rate: 0.0010
Epoch 2/15
 1/85 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.8750 - auc: 0.8765 - loss: 0.4606 - precision: 0.9091 - recall: 0.7692

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.8750 - auc: 0.8765 - loss: 0.4606 - precision: 0.9091 - recall: 0.7692 - val_accuracy: 0.6821 - val_auc: 0.7829 - val_loss: 0.5410 - val_precision: 0.6169 - val_recall: 0.6270 - learning_rate: 0.0010
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 8s 95ms/step - accuracy: 0.8150 - auc: 0.8756 - loss: 0.4478 - precision: 0.7858 - recall: 0.7576 - val_accuracy: 0.6547 - val_auc: 0.8034 - val_loss: 0.5517 - val_precision: 0.5644 - val_recall: 0.7541 - learning_rate: 0.0010
Epoch 4/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.6875 - auc: 0.7571 - loss: 0.5507 - precision: 0.6154 - recall: 0.6154 - val_accuracy: 0.6581 - val_auc: 0.8038 - val_loss: 0.5471 - val_precision: 0.5688 - val_recall: 0.7459 - learning_rate: 0.0010
Epoch 5/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 8s 96ms/step - accuracy: 0.8424 - auc: 0.9109 - loss: 0.3931 - precision: 0.8217 - recall: 0.7892 - val_accuracy: 0.7350 - val_auc: 0.8040 - val_loss: 0.5129 - val_precision: 0.6

## 6 · Curvas de entrenamiento (¿sobreajuste?)

In [12]:
def plot_history(histories, title):
    h1, h2 = histories
    def cat(key):
        v = list(h1.history.get(key, []))
        if h2 is not None:
            v += list(h2.history.get(key, []))
        return v
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(cat("loss"), label="train"); ax[0].plot(cat("val_loss"), label="val")
    ax[0].set_title(f"{title} — pérdida"); ax[0].legend(); ax[0].set_xlabel("epoch")
    ax[1].plot(cat("auc"), label="train"); ax[1].plot(cat("val_auc"), label="val")
    ax[1].set_title(f"{title} — AUC"); ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.tight_layout()
    plt.savefig(REPO / f"reports/figures/curvas_{title.lower().replace(' ','_')}.png",
                dpi=120, bbox_inches="tight")
    plt.show()

plot_history(mnet_hist, "MobileNetV2")
plot_history(eff_hist, "EfficientNetB0")
plot_history(cnn_hist, "CNN")

## 7 · Evaluación comparativa en test

In [13]:
modelos = [
    ("CNN desde cero", str(REPO / "models/best_cnn.keras")),
    ("MobileNetV2",    str(REPO / "models/best_mobilenet.keras")),
    ("EfficientNetB0", str(REPO / "models/best_efficientnet.keras")),
]

filas = []
for nombre, ruta in modelos:
    m = keras.models.load_model(ruta)
    prefix = pathlib.Path(ruta).stem
    met = evaluate.evaluate(m, test_ds, threshold=0.5, prefix=prefix)
    met["modelo"] = nombre
    met["tiempo_inf_s"] = round(evaluate.measure_inference_time(m), 3)
    met["tamaño_MB"] = round(evaluate.model_size_mb(ruta), 1)
    filas.append(met)

df = pd.DataFrame(filas).set_index("modelo")[
    ["accuracy", "recall", "specificity", "precision", "f1", "auc",
     "false_negatives", "false_positives", "tiempo_inf_s", "tamaño_MB"]
]
df.to_csv(REPO / "reports/metrics_comparacion.csv")
df.round(3)

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
              precision    recall  f1-score   support

      normal      0.644     0.982     0.778       328
    fractura      0.905     0.243     0.383       235

    accuracy                          0.673       563
   macro avg      0.774     0.612     0.580       563
weighted avg      0.753     0.673     0.613       563

AUC-ROC: 0.820 | Especificidad: 0.982 | Umbral: 0.500
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 155ms/step
              precision    recall  f1-score   support

      normal      0.850     0.829     0.840       328
    fractura      0.770     0.796     0.782       235

    accuracy                          0.815       563
   macro avg      0.810     0.813     0.811       563
weighted avg      0.816     0.815     0.816       563

AUC-ROC: 0.910 | Especificidad: 0.829 | Umbral: 0.500
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 243ms/step
              precision    recall  f1-score   support

      normal      0.794     0.881     0.835       328
    fra

,accuracy,recall,specificity,precision,f1,auc,false_negatives,false_positives,tiempo_inf_s,tamaño_MB
modelo,,,,,,,,,,
CNN desde cero,0.673,0.243,0.982,0.905,0.383,0.820,178,6,0.065,1.2
MobileNetV2,0.815,0.796,0.829,0.770,0.782,0.910,48,56,0.070,9.7
EfficientNetB0,0.798,0.681,0.881,0.804,0.737,0.919,75,39,0.076,43.4


## 8 · Ajuste del umbral de decisión (paso clínico)

In [14]:
best = keras.models.load_model(str(REPO / "models/best_mobilenet.keras"))
y_true, y_prob = evaluate.get_true_prob(best, test_ds)

for objetivo in (0.90, 0.95, 0.98):
    thr = evaluate.threshold_for_recall(y_true, y_prob, objetivo)
    print(f"recall>={objetivo}: umbral = {thr:.3f}")

UMBRAL = evaluate.threshold_for_recall(y_true, y_prob, 0.95)
print("\nEvaluación de MobileNetV2 con el umbral elegido:")
evaluate.evaluate(best, test_ds, threshold=UMBRAL, prefix="mobilenet_umbral")
print(f"\nUMBRAL elegido = {round(UMBRAL, 3)}  -> cópialo a app/app_gradio.py y resultados.md")

18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 122ms/step
recall>=0.9: umbral = 0.387
recall>=0.95: umbral = 0.310
recall>=0.98: umbral = 0.223

Evaluación de MobileNetV2 con el umbral elegido:
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
              precision    recall  f1-score   support

      normal      0.951     0.649     0.772       328
    fractura      0.661     0.953     0.780       235

    accuracy                          0.776       563
   macro avg      0.806     0.801     0.776       563
weighted avg      0.830     0.776     0.775       563

AUC-ROC: 0.910 | Especificidad: 0.649 | Umbral: 0.310

UMBRAL elegido = 0.31  -> cópialo a app/app_gradio.py y resultados.md


## 9 · Descargar el modelo

Descarga `best_mobilenet.keras` para el despliegue local/Space, **o** súbelo
directamente a un *Model repo* de Hugging Face (recomendado para el Space).

In [ ]:
if _in_colab:
    from google.colab import files
    files.download(str(REPO / 'models/best_mobilenet.keras'))
else:
    print('Local: modelo en', REPO / 'models/best_mobilenet.keras')

# --- Subir a Hugging Face (recomendado para el Space) ---
# !pip -q install huggingface_hub
# from huggingface_hub import login, upload_file
# login()   # pega tu token de HF
# upload_file(
#     path_or_fileobj=str(REPO / 'models/best_mobilenet.keras'),
#     path_in_repo='best_mobilenet.keras',
#     repo_id='stevenrq8/fracturas-modelo',
#     repo_type='model',
# )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

: 

## 10 · Siguientes pasos

1. Vuelca los números de la sección 7 y el umbral (sección 8) en
   **`reports/resultados.md`** y escribe la reflexión crítica.
2. Copia el `UMBRAL` a `app/app_gradio.py`, `app/app_streamlit.py` y `space/app.py`.
3. Publica el Space en Hugging Face (carpeta `space/`).